In [3]:
# 데이터 생성 : csv파일을 불러옴
import pandas as pd
df = pd.read_csv('sentiment_data.csv')

In [7]:
# 독립, 종속변수 분리
X = df['sentence']
y = df['label']

In [10]:
# 훈련, 테스트 분리
# from sklearn.model_selection import train_test_split
# X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)
DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE: ]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE: ]

In [20]:
# 형태소 분리 시 모든 형태소를 포함
from konlpy.tag import Okt
cleaned_sentence = []
def get_preprocessing(sentence):
	okt = Okt()
	result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단, 원형으로
	words = [word for word, pos in result]
	return " ".join(words)
# 분리한 단어들을 합침
# print(X_train[0])
# get_preprocessing(X_train[0])
X_train = X_train.apply(get_preprocessing)
X_test = X_test.apply(get_preprocessing)

# 벡터화
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode="multi_hot"
)
vectorize_layer.adapt(X_train.tolist())
# 모델 설계
import tensorflow as tf
model = models.Sequential([
	layers.Input((1, ), dtype=tf.string),
	vectorize_layer,
	# 히든층
	layers.Dense(64, activation='relu'),
	layers.Dense(32, activation='relu'),
	layers.Dense(1, activation='sigmoid'),
])

# 모델 설정
model.compile(optimizer='adam', 
							loss='binary_crossentropy',
							metrics=['accuracy']
)


In [21]:
# 학습
history = model.fit(X_train.values, y_train, epochs=100, verbose=1, validation_split=0.2,batch_size=32)


Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5531 - loss: 0.6893 - val_accuracy: 0.7563 - val_loss: 0.6524
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8656 - loss: 0.6159 - val_accuracy: 0.9187 - val_loss: 0.5815
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9688 - loss: 0.5193 - val_accuracy: 0.9937 - val_loss: 0.4625
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9953 - loss: 0.3733 - val_accuracy: 1.0000 - val_loss: 0.2952
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9984 - loss: 0.2120 - val_accuracy: 1.0000 - val_loss: 0.1501
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0999 - val_accuracy: 1.0000 - val_loss: 0.0684
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0469 - val_accuracy: 1.0000 - val_loss: 0.0352
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0254 - val_accuracy: 1.0000 - 

In [25]:
# 평가
_, test_acc = model.evaluate(X_test.values,y_test)
print(f'정확도 : {test_acc}')


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 6.4223e-05  
정확도 : 1.0


In [ ]:
# 예측
# "너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요"
# 오늘 날이 좋고 기분이 좋아요
# 듣고 있는 음악이 슬퍼서 우울해요
import numpy as np
sentences = [
	"음악 선곡이 가슴을 울리고 배우들의 몰입감이 엄청나며 연출력이 세계적은 수준이다.",
]

predictions = model.predict(np.array(sentences,dtype=object))
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step


array([[0.81745374],
       [0.23103659],
       [0.44146067]], dtype=float32)